# Factor sensitivity and risk decomposition

**Prerequisites:** notebooks **05** (pricing fundamentals) and **08** (risk and factor analytics).  This deep-dive covers the `finstack.valuations` **factor-model** API — computing how a portfolio of instruments responds to structured risk-factor shocks, and decomposing forecasted portfolio risk into factor and position contributions.

Topics covered:

- Defining **positions** (instrument JSON + weight)
- Defining **risk factors** (`FactorDefinition` with `MarketMapping`)
- **`compute_factor_sensitivities`** — first-order (delta) sensitivities via central differences
- **`compute_pnl_profiles`** — scenario P&L via full repricing across a factor grid
- **`decompose_factor_risk`** — parametric Euler decomposition of portfolio variance/volatility/VaR/ES
- `SensitivityMatrix`, `FactorPnlProfile`, and `RiskDecomposition` wrapper classes
- DataFrame export for downstream analytics

## Concepts

Factor sensitivity answers: **"how much does my portfolio P&L change when a risk factor moves?"**

The pipeline works as follows:

1. **Positions** — a list of `(id, instrument, weight)` tuples serialized as JSON.  Each instrument is the same tagged JSON used by `price_instrument`.
2. **Factors** — `FactorDefinition` objects mapping a named risk factor to concrete market bumps (parallel curve shift, equity spot move, FX rate change, etc.).
3. **Engine** — either `DeltaBasedEngine` (finite-difference deltas) or `FullRepricingEngine` (scenario grid with full repricing at each shift).
4. **Result** — a `SensitivityMatrix` (positions × factors) or a list of `FactorPnlProfile` (per-factor scenario P&L).

Both engines use `BumpSizeConfig` to control the magnitude of bumps per factor type (default: 1 bp for rates/credit, 1 % for equity/FX).

### Imports

In [1]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, MarketContext, PriceCurve
from finstack.valuations import (
    compute_factor_sensitivities,
    compute_pnl_profiles,
    decompose_factor_risk,
)

print("Imported factor-model API.")

Imported factor-model API.


## 1. Market and instrument setup

We build a small market with a USD discount curve and a spot price, then define two instruments:
- A 5-year fixed-rate bond discounted off `USD-OIS`
- A 6-month deposit also on `USD-OIS`

In [2]:
as_of = "2025-01-15"

mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (5.0, 0.75), (10.0, 0.55)],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()

bond_5y = {
    "type": "bond",
    "spec": {
        "id": "BOND-5Y",
        "notional": {"amount": 1_000_000.0, "currency": "USD"},
        "issue_date": "2025-01-15",
        "maturity": "2030-01-15",
        "discount_curve_id": "USD-OIS",
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "rate": 0.05,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "stub": "None",
                "end_of_month": False,
                "payment_lag_days": 0,
            }
        },
        "attributes": {},
    },
}

deposit_6m = {
    "type": "deposit",
    "spec": {
        "id": "DEP-6M",
        "notional": {"amount": 1_000_000.0, "currency": "USD"},
        "quote_rate": 0.05,
        "start_date": "2025-01-15",
        "maturity": "2025-07-15",
        "day_count": "Act360",
        "discount_curve_id": "USD-OIS",
        "attributes": {},
    },
}

positions_json = json.dumps([
    {"id": "bond_5y", "instrument": bond_5y, "weight": 1.0},
    {"id": "dep_6m", "instrument": deposit_6m, "weight": 1.0},
])

print(f"Defined 2 positions against a {len(market_json)}-char market snapshot.")

Defined 2 positions against a 808-char market snapshot.


## 2. Defining risk factors

A `FactorDefinition` has four fields:

| Field | Description |
|-------|-------------|
| `id` | Unique factor name (e.g. `"usd_rates"`) |
| `factor_type` | Category: `Rates`, `Credit`, `Equity`, `FX`, `Volatility`, etc. |
| `market_mapping` | How the factor translates to market bumps |
| `description` | Optional documentation string |

**`MarketMapping` variants:**

- `CurveParallel` — parallel shift on one or more curves (units: `rate_bp`, `percent`, etc.)
- `CurveBucketed` — triangular key-rate bumps at specific tenors
- `EquitySpot` — percentage move on equity/commodity tickers
- `FxRate` — FX pair percentage move
- `VolShift` — parallel vol surface shift

In [3]:
factors_json = json.dumps([
    {
        "id": "usd_rates",
        "factor_type": "Rates",
        "market_mapping": {
            "CurveParallel": {
                "curve_ids": ["USD-OIS"],
                "units": "rate_bp",
            }
        },
        "description": "Parallel USD rates shift (1 bp)",
    },
])

print("Defined 1 factor: usd_rates (parallel curve shift).")

Defined 1 factor: usd_rates (parallel curve shift).


## 3. Delta-based sensitivities

`compute_factor_sensitivities` uses **central finite differences**: it bumps the market up and down by the configured amount (default 1 bp for rates), reprices each position, and returns `(PV_up - PV_down) / (2 * bump)` weighted by position weight.

The result is a `SensitivityMatrix` with positions on the row axis and factors on the column axis.

In [4]:
matrix = compute_factor_sensitivities(
    positions_json, factors_json, market_json, as_of
)

print(matrix)
print(f"Position IDs: {matrix.position_ids}")
print(f"Factor IDs:   {matrix.factor_ids}")
print()

for i, pid in enumerate(matrix.position_ids):
    delta = matrix.delta(i, 0)
    print(f"  {pid}: delta = {delta:,.2f} (PV change per 1bp rates move)")

SensitivityMatrix(positions=2, factors=1)
Position IDs: ['bond_5y', 'dep_6m']
Factor IDs:   ['usd_rates']

  bond_5y: delta = -431.73 (PV change per 1bp rates move)
  dep_6m: delta = -49.58 (PV change per 1bp rates move)


### DataFrame export

The `SensitivityMatrix` exports directly to a pandas DataFrame with positions as the index and factors as columns — ready for heatmaps, aggregation, or portfolio-level risk reports.

In [5]:
df = matrix.to_dataframe()
print(df)
print()
print(f"Total portfolio delta: {df['usd_rates'].sum():,.2f}")

          usd_rates
bond_5y -431.725845
dep_6m   -49.575211

Total portfolio delta: -481.30


## 4. Scenario P&L profiles (full repricing)

`compute_pnl_profiles` reprices each position at multiple factor shift points along a symmetric grid.  With the default `n_scenario_points=5` and a 1 bp bump, the grid is `[-2, -1, 0, +1, +2]` multiples of the bump size.

This captures **non-linear behavior** (gamma/convexity) that delta alone misses.

In [6]:
profiles = compute_pnl_profiles(
    positions_json, factors_json, market_json, as_of,
    n_scenario_points=9,
)

print(f"Computed {len(profiles)} profile(s):")
for p in profiles:
    print(f"  {p}")

Computed 1 profile(s):
  FactorPnlProfile(factor="usd_rates", shifts=9, positions=2)


### Profile DataFrame

Each `FactorPnlProfile` exports to a DataFrame with scenario shifts as the index and position P&L as columns.  The center row (shift = 0) is always zero (no bump = no P&L).

In [7]:
position_ids = ["bond_5y", "dep_6m"]
profile = profiles[0]

pdf = profile.to_dataframe(position_ids)
pdf.index.name = "shift (x bump_size)"
pdf["portfolio"] = pdf.sum(axis=1)
print(f"Factor: {profile.factor_id}")
print(pdf.to_string(float_format="{:,.2f}".format))

Factor: usd_rates
                      bond_5y  dep_6m  portfolio
shift (x bump_size)                             
-4.0                 1,728.56  198.32   1,926.88
-3.0                 1,296.11  148.74   1,444.85
-2.0                   863.87   99.16     963.02
-1.0                   431.83   49.58     481.41
 0.0                     0.00    0.00       0.00
 1.0                  -431.62  -49.57    -481.20
 2.0                  -863.04  -99.15    -962.18
 3.0                -1,294.25 -148.71  -1,442.96
 4.0                -1,725.25 -198.28  -1,923.53


## 5. Custom bump configuration

The default `BumpSizeConfig` uses 1 bp for rates and credit, 1 % for equity and FX.  You can override per factor type or per factor ID.

In [8]:
bump_config = json.dumps({
    "rates_bp": 10.0,
    "credit_bp": 5.0,
    "equity_pct": 1.0,
    "fx_pct": 1.0,
    "vol_points": 1.0,
})

matrix_10bp = compute_factor_sensitivities(
    positions_json, factors_json, market_json, as_of,
    bump_config_json=bump_config,
)

df_10bp = matrix_10bp.to_dataframe()
print("Deltas at 10 bp bump:")
print(df_10bp)
print()
print("Compare with 1 bp default:")
print(df)

Deltas at 10 bp bump:
          usd_rates
bond_5y -431.727518
dep_6m   -49.575213

Compare with 1 bp default:
          usd_rates
bond_5y -431.725845
dep_6m   -49.575211


## 6. Multi-factor example: rates + bucketed key-rate

Real risk reports decompose exposure across multiple factors.  Here we add a 5-year key-rate factor alongside the parallel shift.

In [9]:
multi_factors = json.dumps([
    {
        "id": "rates_parallel",
        "factor_type": "Rates",
        "market_mapping": {
            "CurveParallel": {
                "curve_ids": ["USD-OIS"],
                "units": "rate_bp",
            }
        },
    },
    {
        "id": "rates_5y_key",
        "factor_type": "Rates",
        "market_mapping": {
            "CurveBucketed": {
                "curve_id": "USD-OIS",
                "tenor_weights": [[2.0, 0.25], [5.0, 1.0], [10.0, 0.25]],
            }
        },
        "description": "5y key rate bump with triangular weights",
    },
])

multi_matrix = compute_factor_sensitivities(
    positions_json, multi_factors, market_json, as_of
)

print("Multi-factor sensitivity matrix:")
print(multi_matrix.to_dataframe())

Multi-factor sensitivity matrix:
         rates_parallel  rates_5y_key
bond_5y     -431.725845      1.161686
dep_6m       -49.575211      0.000000


## 7. Portfolio risk decomposition

With sensitivities computed, we can now ask: **what is the forecasted portfolio volatility and how much does each factor (and each position) contribute?**

`decompose_factor_risk` takes the `SensitivityMatrix` from above, a **factor covariance matrix** (annual variances and covariances of factor returns), and a risk measure.  It uses the **parametric Euler decomposition**:

$$\text{Portfolio variance} = \mathbf{e}^\top \boldsymbol{\Sigma} \, \mathbf{e}$$

where $\mathbf{e}$ is the vector of weighted portfolio exposures (column sums of the sensitivity matrix) and $\boldsymbol{\Sigma}$ is the factor covariance matrix.

Euler allocation then attributes total risk to each factor $k$ via:

$$\text{Factor}_k\text{ contribution} = e_k \cdot (\boldsymbol{\Sigma} \, \mathbf{e})_k$$

and to each position × factor pair via:

$$\text{Position}_{i,k}\text{ contribution} = \delta_{i,k} \cdot (\boldsymbol{\Sigma} \, \mathbf{e})_k$$

Supported risk measures: `"variance"`, `"volatility"`, `{"var": {"confidence": 0.99}}`, `{"expected_shortfall": {"confidence": 0.975}}`.

### Single-factor variance decomposition

We'll use the single-factor `usd_rates` sensitivity matrix from section 3, and supply a trivial 1×1 covariance matrix (the annual variance of the rates factor).

In [10]:
# Single-factor covariance: annual variance of 100 bp^2 (i.e. 10 bp vol)
cov_1f = json.dumps({
    "factor_ids": ["usd_rates"],
    "n": 1,
    "data": [100.0],
})

decomp = decompose_factor_risk(matrix, cov_1f)
print(f"Risk measure:  {decomp.measure}")
print(f"Total risk:    {decomp.total_risk:,.2f}  (portfolio variance in PV terms)")
print(f"Residual risk: {decomp.residual_risk:.4f}")
print()

print("Factor contributions:")
for fc in decomp.factor_contributions():
    print(f"  {fc['factor_id']:20s}  absolute={fc['absolute_risk']:12,.2f}  "
          f"relative={fc['relative_risk']:.2%}  marginal={fc['marginal_risk']:12,.2f}")
print()

print("Position × factor contributions:")
for pfc in decomp.position_factor_contributions():
    print(f"  {pfc['position_id']:12s} × {pfc['factor_id']:20s}  "
          f"risk_contribution={pfc['risk_contribution']:12,.2f}")

Risk measure:  Variance
Total risk:    23,165,070.70  (portfolio variance in PV terms)
Residual risk: 0.0000

Factor contributions:
  usd_rates             absolute=23,165,070.70  relative=100.00%  marginal=  -48,130.11

Position × factor contributions:
  bond_5y      × usd_rates             risk_contribution=20,779,010.55
  dep_6m       × usd_rates             risk_contribution=2,386,060.15


### Switching risk measures

The same sensitivities and covariance can be decomposed under different risk measures.  Let's compare variance, volatility, and 99% VaR.

In [11]:
import pandas as pd

measures = [
    ("Variance",           '"variance"'),
    ("Volatility",         '"volatility"'),
    ("VaR 99%",            '{"var": {"confidence": 0.99}}'),
    ("ES 97.5%",           '{"expected_shortfall": {"confidence": 0.975}}'),
]

rows = []
for label, rm_json in measures:
    d = decompose_factor_risk(matrix, cov_1f, rm_json)
    rows.append({"measure": label, "total_risk": d.total_risk})

print("Risk measure comparison (single factor, same portfolio):")
print(pd.DataFrame(rows).to_string(index=False))

Risk measure comparison (single factor, same portfolio):
   measure   total_risk
  Variance 2.316507e+07
Volatility 4.813011e+03
   VaR 99% 1.119674e+04
  ES 97.5% 1.125187e+04


### Multi-factor risk decomposition with correlation

Now let's use the two-factor sensitivity matrix from section 6 and supply a 2×2 covariance matrix with non-zero correlation between the parallel and key-rate factors.

In [12]:
# 2x2 covariance: parallel (100 bp^2), key-rate (25 bp^2), cross-cov 30
cov_2f = json.dumps({
    "factor_ids": ["rates_parallel", "rates_5y_key"],
    "n": 2,
    "data": [100.0, 30.0,
             30.0,  25.0],
})

decomp_2f = decompose_factor_risk(multi_matrix, cov_2f, '"volatility"')
print(f"Total portfolio volatility: {decomp_2f.total_risk:,.2f}")
print()

print("Factor contributions (DataFrame):")
print(decomp_2f.to_factor_dataframe().to_string(index=False))
print()

print("Position × factor contributions (DataFrame):")
print(decomp_2f.to_position_factor_dataframe().to_string(index=False))

Total portfolio volatility: 4,809.53

Factor contributions (DataFrame):
     factor_id  absolute_risk  relative_risk  marginal_risk
rates_parallel    4813.008319       1.000724      -9.999995
  rates_5y_key      -3.480566      -0.000724      -2.996134

Position × factor contributions (DataFrame):
position_id      factor_id  risk_contribution
    bond_5y rates_parallel        4317.256439
    bond_5y   rates_5y_key          -3.480566
     dep_6m rates_parallel         495.751880
     dep_6m   rates_5y_key          -0.000000


### Interpreting the decomposition

Key properties of the Euler allocation:
- **absolute_risk** values sum exactly to `total_risk` (plus `residual_risk`).
- **relative_risk** values sum to 1.0 when residual is zero.
- **marginal_risk** is the derivative of portfolio risk with respect to the factor exposure — useful for portfolio optimization.
- **position × factor** contributions let you drill from portfolio risk down to the exact instrument and risk driver.

The `RiskDecomposition` object provides both list-of-dict and DataFrame exports for each view.

## Takeaways

- **`compute_factor_sensitivities`** gives first-order (delta) sensitivities via central finite differences — fast and suitable for linear risk.
- **`compute_pnl_profiles`** provides the full scenario P&L surface for non-linear analysis (gamma, convexity).
- **`decompose_factor_risk`** combines sensitivities with a factor covariance matrix to produce forecasted portfolio risk and Euler-allocated contributions by factor and position.
- **Risk measures**: `"variance"`, `"volatility"`, `{"var": {"confidence": 0.99}}`, `{"expected_shortfall": {"confidence": 0.975}}` are all supported.
- **`SensitivityMatrix.to_dataframe()`**, **`FactorPnlProfile.to_dataframe()`**, and **`RiskDecomposition.to_factor_dataframe()`** / **`.to_position_factor_dataframe()`** integrate directly with pandas.
- **`BumpSizeConfig`** lets you customize bump magnitudes per factor type or per factor ID.

**Next:** see notebook **08** (risk and factor analytics) for return-based factor regressions and VaR, or **12** (scenarios and stress testing) for macro scenario application.